# Membangun Aplikasi Generasi Gambar (OpenAI)

Ada lebih dari sekadar generasi teks pada LLM. Anda juga dapat menghasilkan gambar dari deskripsi teks. Gambar sebagai modalitas berguna di berbagai bidang seperti MedTech, arsitektur, pariwisata, pengembangan game, pemasaran, dan lainnya. Dalam pelajaran ini, kami bekerja dengan model **GPT Image** terbaru di platform OpenAI.

## Tujuan pembelajaran

Di akhir pelajaran ini Anda akan bisa:

- Menjelaskan apa itu generasi gambar dan di mana kegunaannya.
- Memahami keluarga model `gpt-image` dan bagaimana perbedaannya dengan model DALL·E lama.
- Membangun aplikasi generasi gambar dan mengedit gambar.

## Apa itu generasi gambar?

Model generasi gambar membuat gambar dari sebuah prompt teks. Model modern seperti `gpt-image` mempelajari hubungan antara teks dan gambar selama pelatihan, kemudian secara iteratif mengubah kebisingan acak menjadi gambar yang sesuai dengan prompt Anda.

Dua keluarga model gambar yang terkenal adalah:

- **`gpt-image` (OpenAI)** — generasi saat ini yang digunakan dalam pelajaran ini. Mendukung generasi teks-ke-gambar dan pengeditan gambar (inpainting dengan masker).
- **Midjourney** — model pihak ketiga populer dengan layanan sendiri dan alur kerja berbasis Discord.

> Model gambar OpenAI yang lebih lama — **DALL·E 2** dan **DALL·E 3** — adalah warisan. DALL·E 3 tidak lagi tersedia untuk deployment baru, dan fitur seperti `create_variation` hanya ada di DALL·E 2. Gunakan model `gpt-image` untuk aplikasi baru.

> **Penting:** model `gpt-image` mengembalikan gambar yang dihasilkan sebagai **base64** (`b64_json`), bukan sebagai URL. Kode Anda mendekode string base64 menjadi byte dan menyimpannya — tidak ada URL gambar untuk diunduh.


## Membangun aplikasi generasi gambar pertamamu

Jadi, apa yang dibutuhkan untuk membangun aplikasi generasi gambar? Kamu memerlukan perpustakaan berikut:

- **python-dotenv**, sangat disarankan menggunakan perpustakaan ini untuk menyimpan rahasiamu dalam file *.env* terpisah dari kode.
- **openai**, perpustakaan ini yang akan kamu gunakan untuk berinteraksi dengan API OpenAI.
- **pillow**, untuk bekerja dengan gambar di Python.


1. Buat file *.env* dengan isi berikut:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Kumpulkan library di atas dalam file bernama *requirements.txt* seperti berikut:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Selanjutnya, buat virtual environment dan instal library tersebut:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Untuk Windows, gunakan perintah berikut untuk membuat dan mengaktifkan lingkungan virtual Anda:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tambahkan kode berikut dalam file yang disebut *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Buat objek OpenAI (membaca OPENAI_API_KEY dari .env Anda)
    client = openai.OpenAI()


    try:
        # Buat gambar dengan menggunakan API generasi gambar
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks prompt Anda di sini
            size='1024x1024',
            n=1
        )
        # Atur direktori untuk menyimpan gambar
        image_dir = os.path.join(os.curdir, 'images')

        # Jika direktori tidak ada, buatlah
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inisialisasi jalur gambar (perhatikan tipe file harus png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # model gpt-image mengembalikan gambar sebagai base64 (b64_json), bukan URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Tampilkan gambar di penampil gambar default
        image = Image.open(image_path)
        image.show()

    # tangkap pengecualian
    except openai.BadRequestError as err:
        print(err)

    ```

Mari jelaskan kode ini:

- Pertama, kita mengimpor perpustakaan yang kita butuhkan, termasuk perpustakaan OpenAI, perpustakaan dotenv, modul base64, dan perpustakaan Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Setelah itu, kita membuat klien, yang membaca kunci API dari ``.env`` Anda.

    ```python
    # Buat objek OpenAI
    client = openai.OpenAI()
    ```

- Selanjutnya, kita menghasilkan gambar:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks prompt Anda di sini
        size='1024x1024',
        n=1
    )
    ```

    Model `gpt-image` mengembalikan gambar sebagai string **base64** di `data[0].b64_json`. Kita mendekodenya menjadi byte dan menulisnya ke file — tidak ada URL untuk diunduh.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Terakhir, kita membuka gambar dan menggunakan penampil gambar standar untuk menampilkannya:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Detail lebih lanjut tentang menghasilkan gambar

Mari kita lihat parameter dari `images.generate`:

- **model**, adalah model gambar yang digunakan (misalnya `gpt-image-1`).
- **prompt**, adalah teks prompt yang digunakan untuk menghasilkan gambar. Di sini adalah "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, adalah ukuran dari gambar yang dihasilkan (misalnya `1024x1024`, `1536x1024`, `1024x1536`, atau `"auto"`).
- **n**, adalah jumlah gambar yang dihasilkan. Di sini kita menghasilkan satu.

> Model gambar tidak menggunakan parameter `temperature` — itu adalah kontrol untuk generasi teks. Untuk mendapatkan variasi, panggil API lagi; untuk mengurangi variasi, buat prompt Anda lebih spesifik.

## Kapabilitas tambahan dari generasi gambar

Anda telah melihat bagaimana menghasilkan gambar dengan beberapa baris kode Python. Model `gpt-image` juga dapat **mengedit** gambar yang sudah ada. Dengan menyediakan gambar, **mask** opsional (yang menandai area yang akan diubah), dan sebuah prompt, Anda dapat mengubah sebagian gambar — misalnya, menambahkan topi pada kelinci kita.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edit juga dikembalikan sebagai base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Gambar dasar hanya berisi kelinci; gambar akhir memiliki topi di kelinci tersebut.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
